# 深度学习优化器总结

优化器是神经网络训练的「方向盘」——它决定了模型参数如何根据梯度进行更新。不同的优化器就像不同的驾驶策略，有的激进、有的稳健、有的能自适应路况。下面按演进顺序逐一介绍。

---

## a. SGD（随机梯度下降）

### 公式与参数

$\theta_{t+1} = \theta_t - \eta \cdot g_t$

| 参数 | 含义 |
|------|------|
| $\theta_t$ | 第 $t$ 步的模型参数 |
| $\eta$ | 学习率（learning rate） |
| $g_t = \nabla_\theta L(\theta_t)$ | 当前 batch 的梯度 |

### 解决了什么问题

在 SGD 之前，人们用 **BGD（批量梯度下降）**：每次更新要把整个数据集算一遍梯度。数据一大根本跑不动。SGD 的思想很简单——**每次只拿一个小 batch（甚至一个样本）算梯度就更新**，速度快、能在线学习，而且梯度里的随机噪声反而有助于跳出局部最优。

In [ ]:
for x, y in dataloader:
    loss = loss_fn(model(x), y)
    loss.backward()
    for p in model.parameters():
        p.data -= lr * p.grad.data

---

## b. Momentum（动量法）

### 公式与参数

$v_{t} = \beta v_{t-1} + (1 - \beta) g_t$

$\theta_{t+1} = \theta_t - \eta \cdot v_t$

| 参数 | 含义 |
|------|------|
| $v_t$ | 第 $t$ 步的速度（动量项） |
| $\beta$ | 动量系数，通常取 0.9 |
| $\eta$ | 学习率 |

### 解决了什么问题

SGD 有两个痛点：

1. **峡谷震荡**：在某个方向梯度很大、另一个方向梯度很小时，SGD 会在陡峭方向上反复横跳，收敛慢。
2. **局部最小/鞍点**：梯度为零时 SGD 直接卡住。

Momentum 的想法来自物理学的**惯性**——把之前累积的速度方向**指数加权平均**进来。就像推一个铁球下坡，越滚越快，遇到小坑也能靠惯性冲过去。$\beta$ 控制「惯性多大」，越大惯性越强（换个角度看，你愿意相信历史方向的「记忆」有多长）。

In [ ]:
v = [0.0] * len(params)  # 初始化速度

for x, y in dataloader:
    loss = loss_fn(model(x), y)
    loss.backward()
    for i, p in enumerate(model.parameters()):
        v[i] = beta * v[i] + (1 - beta) * p.grad.data
        p.data -= lr * v[i]

---

## c. NAG（Nesterov 加速梯度）

### 公式与参数

$\tilde{\theta}_t = \theta_t - \eta \cdot \beta v_{t-1}$

$v_{t} = \beta v_{t-1} + (1 - \beta) g_t(\tilde{\theta}_t)$

$\theta_{t+1} = \theta_t - \eta \cdot v_t$

| 参数 | 含义 |
|------|------|
| $\tilde{\theta}_t$ | 「前瞻」位置 |
| $g_t(\tilde{\theta}_t)$ | 在 $\tilde{\theta}_t$ 处计算的梯度 |
| $\beta$ | 动量系数 |

### 解决了什么问题

Momentum 有个问题：它是「先冲出去，到了位置再看梯度」。如果前面是悬崖，等你发现已经晚了。

NAG 的改进思路很巧妙——**先按惯性走一步，在那个位置看一眼，发现不对赶紧调**。具体做法是：先走到 $\tilde{\theta}_t$（纯靠之前的动量），然后在 $\tilde{\theta}_t$ 处算梯度来修正方向。这就像打篮球时的「预判跑位」——你不太会往对手已经站好的位置冲，而是提前调整路线。

In [ ]:
v = [0.0] * len(params)

for x, y in dataloader:
    # 先「前瞻」一步
    for i, p in enumerate(model.parameters()):
        p.data -= lr * beta * v[i]

    # 在前瞻位置算梯度
    loss = loss_fn(model(x), y)
    loss.backward()

    for i, p in enumerate(model.parameters()):
        v[i] = beta * v[i] + (1 - beta) * p.grad.data
        p.data -= lr * (1 - beta) * p.grad.data  # 修正

---

## d. AdaGrad（自适应梯度）

### 公式与参数

$s_t = s_{t-1} + g_t^2 \quad\text{（逐元素平方和）}$

$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_t} + \varepsilon} \cdot g_t$

| 参数 | 含义 |
|------|------|
| $s_t$ | 历史梯度的逐元素平方累加和 |
| $\varepsilon$ | 平滑项，防止除零（通常取 $10^{-8}$） |
| $\eta$ | 全局学习率 |

### 解决了什么问题

之前的优化器对所有参数用同一个学习率，这很粗暴。想象一个场景：有些参数（词嵌入里的高频词）经常被更新，梯度信号多；有些参数（低频词）很少被更新。给它们一样的学习率，高频词永远在震荡，低频词永远学不到——这不公平。

AdaGrad 的核心思想：**更新越频繁的参数，学习率自动衰减得越多；更新越少的参数，学习率保持较大**。实现方式是累加每个参数的历史梯度平方和，用它来除学习率。「用的多就降速，用的少就加速」，相当直觉。

但 AdaGrad 有个致命缺陷：$s_t$ 是单调递增的，训练久了学习率会衰减到零，模型停止学习。

In [ ]:
s = [0.0] * len(params)

for x, y in dataloader:
    loss = loss_fn(model(x), y)
    loss.backward()
    for i, p in enumerate(model.parameters()):
        s[i] += p.grad.data ** 2
        p.data -= (lr / (s[i].sqrt() + eps)) * p.grad.data

---

## e. RMSProp（均方根传播）

### 公式与参数

$s_t = \beta s_{t-1} + (1 - \beta) g_t^2$

$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_t} + \varepsilon} \cdot g_t$

| 参数 | 含义 |
|------|------|
| $s_t$ | 梯度平方的指数移动平均 |
| $\beta$ | 衰减系数，通常取 0.9 |
| $\varepsilon$ | 平滑项 |

### 解决了什么问题

AdaGrad 的 $s_t$ 只增不减，是它的「阿喀琉斯之踵」。RMSProp 做了一个简单但关键的改动：把**累加**改成**指数移动平均**。$\beta$ 越大，记忆越长，$s_t$ 越像 AdaGrad；$\beta$ 越小，越只看近期。

这就像记笔记——AdaGrad 是从不翻页的笔记本，写到后面全是墨迹；RMSProp 是活页本，旧的页慢慢抽掉，永远只看最近几页。结果就是：**既能自适应地给每个参数调学习率，又不会让学习率单调递减到零**。

In [ ]:
s = [0.0] * len(params)

for x, y in dataloader:
    loss = loss_fn(model(x), y)
    loss.backward()
    for i, p in enumerate(model.parameters()):
        s[i] = beta * s[i] + (1 - beta) * (p.grad.data ** 2)
        p.data -= (lr / (s[i].sqrt() + eps)) * p.grad.data

---

## f. Adam（自适应动量估计）

### 公式与参数

$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$

$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$

$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \quad \text{（偏差修正）}$

$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{\hat{v}_t} + \varepsilon} \cdot \hat{m}_t$

| 参数 | 含义 |
|------|------|
| $m_t$ | 一阶矩（梯度的指数移动平均，动量） |
| $v_t$ | 二阶矩（梯度平方的指数移动平均，自适应学习率） |
| $\beta_1$ | 一阶矩衰减系数，通常取 0.9 |
| $\beta_2$ | 二阶矩衰减系数，通常取 0.999 |
| $\hat{m}_t$, $\hat{v}_t$ | 偏差修正后的一阶、二阶矩 |
| $\eta$ | 学习率 |

### 解决了什么问题

Adam 的名字来自 **Adaptive Moment Estimation**，它就是 **「Momentum + RMSProp」的合体**：

- $m_t$（一阶矩）负责动量——记住「该往哪儿走」；
- $v_t$（二阶矩）负责自适应学习率——知道「每一步迈多大」。

但 Adam 还多了一个细节：**偏差修正**。$m_t$ 和 $v_t$ 初始化为零，前几步的值会系统性地偏小（因为零初始化 + EMA 的「冷启动」问题）。除以 $1 - \beta^t$ 相当于做了个无偏估计的校准，让训练前期更新幅度不会太小。

Adam 几乎是今天深度学习里**默认的首选优化器**——训练快、超参数不敏感、在大部分任务上不用怎么调就能跑出不错的结果。

In [ ]:
m = [0.0] * len(params)
v = [0.0] * len(params)
t = 0

for x, y in dataloader:
    t += 1
    loss = loss_fn(model(x), y)
    loss.backward()
    for i, p in enumerate(model.parameters()):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad.data          # 一阶矩
        v[i] = beta2 * v[i] + (1 - beta2) * (p.grad.data ** 2)   # 二阶矩
        m_hat = m[i] / (1 - beta1 ** t)   # 偏差修正
        v_hat = v[i] / (1 - beta2 ** t)   # 偏差修正
        p.data -= lr * m_hat / (v_hat.sqrt() + eps)

---

## g. AdamW（权重衰减解耦的 Adam）

### 公式与参数

$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$

$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$

$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$

$\theta_{t+1} = \theta_t - \eta \left( \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \varepsilon} + \lambda \theta_t \right)$

| 参数 | 含义 |
|------|------|
| $\lambda$ | **权重衰减系数**（weight decay） |
| 其余同 Adam | — |

### 解决了什么问题

老版本的 Adam 里，人们习惯加 L2 正则化来防止过拟合，实现方式是直接把 $\lambda \|\theta\|^2$ 加到 loss 里。这在 **SGD 下等价于权重衰减**，但在 Adam 这种自适应优化器下**不成立**。

原因在哪？L2 正则化的梯度会让 $v_t$（二阶矩）里混进正则项的信息，导致自适应学习率的调节被污染——每个参数实际受到的衰减力度不一致，大梯度参数衰减少、小梯度参数衰减多，最终泛化效果差。

AdamW 的解法很干净：**把权重衰减从 loss 里拿出来，直接在参数更新时减掉**。这样做的好处是：衰减力度只由 $\lambda$ 决定，不受自适应学习率干扰。实验表明 AdamW 比「Adam + L2」的泛化效果好，而且调参更直观——$\lambda$ 和 $\eta$ 的职责彻底分开。

> 一句话：**SGD 里 L2 正则化 = 权重衰减；Adam 里这俩不是一回事。AdamW 把它俩解耦了。**

In [ ]:
m = [0.0] * len(params)
v = [0.0] * len(params)
t = 0

for x, y in dataloader:
    t += 1
    loss = loss_fn(model(x), y)  # 注意：loss 不加 L2 正则项
    loss.backward()
    for i, p in enumerate(model.parameters()):
        m[i] = beta1 * m[i] + (1 - beta1) * p.grad.data
        v[i] = beta2 * v[i] + (1 - beta2) * (p.grad.data ** 2)
        m_hat = m[i] / (1 - beta1 ** t)
        v_hat = v[i] / (1 - beta2 ** t)
        p.data -= lr * (m_hat / (v_hat.sqrt() + eps) + weight_decay * p.data)
        #                                                    ^^^^^^^^^^^^^^^^
        #                                            权重衰减直接作用在参数上，不经过 v_t

---

## 快速回顾

| 优化器 | 核心思想 | 一句话 |
|--------|----------|--------|
| SGD | 小批量随机梯度 | 最简单直接，噪声帮助跳出局部最优 |
| Momentum | 指数移动平均的动量 | 减少震荡，靠惯性冲过鞍点 |
| NAG | 前瞻 + 修正 | 先预判再调整，弯道不翻车 |
| AdaGrad | 梯度平方累加，逐参数调学习率 | 高频少学、低频多学，但最后学不动 |
| RMSProp | 梯度平方的指数移动平均 | 修复 AdaGrad 学习率归零的问题 |
| Adam | 动量 + RMSProp + 偏差修正 | 全能选手，默认就用它 |
| AdamW | 权重衰减与自适应学习率解耦 | Adam 的正则化修正版，泛化更好 |